In [25]:
import pandas as pd
import numpy as np

# Loading the cleaned ball-by-ball dataset produced from Section 5.1
df = pd.read_csv("data/deliveries_cleaned.csv")

# Inspecting structure and basic sanity
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251456 entries, 0 to 251455
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   matchid           251456 non-null  int64  
 1   inning            251456 non-null  int64  
 2   over              251456 non-null  int64  
 3   ball              251456 non-null  int64  
 4   batting_team      251456 non-null  object 
 5   bowling_team      251456 non-null  object 
 6   batsman           251456 non-null  object 
 7   non_striker       251456 non-null  object 
 8   bowler            251456 non-null  object 
 9   batsman_runs      251456 non-null  int64  
 10  extras            251456 non-null  int64  
 11  iswide            251456 non-null  float64
 12  isnoball          251456 non-null  float64
 13  byes              251456 non-null  float64
 14  legbyes           251456 non-null  float64
 15  penalty           251456 non-null  float64
 16  dismissal_kind    25

,matchid,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,iswide,isnoball,byes,legbyes,penalty,dismissal_kind,player_dismissed,date,total_runs,is_wicket
0,335982,1,0,1,kolkata knight riders,royal challengers bangalore,sc ganguly,bb mccullum,p kumar,0,...,0.0,0.0,0.0,1.0,0.0,none,none,2008-04-18,2,0
1,335982,1,0,2,kolkata knight riders,royal challengers bangalore,bb mccullum,sc ganguly,p kumar,0,...,0.0,0.0,0.0,0.0,0.0,none,none,2008-04-18,0,0
2,335982,1,0,3,kolkata knight riders,royal challengers bangalore,bb mccullum,sc ganguly,p kumar,0,...,1.0,0.0,0.0,0.0,0.0,none,none,2008-04-18,1,0
3,335982,1,0,4,kolkata knight riders,royal challengers bangalore,bb mccullum,sc ganguly,p kumar,0,...,0.0,0.0,0.0,0.0,0.0,none,none,2008-04-18,0,0
4,335982,1,0,5,kolkata knight riders,royal challengers bangalore,bb mccullum,sc ganguly,p kumar,0,...,0.0,0.0,0.0,0.0,0.0,none,none,2008-04-18,0,0


In [26]:
# Sorting deliveries chronologically to prevent future information leakage
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(
    by=["date", "matchid", "inning", "over", "ball"]
).reset_index(drop=True)

In [27]:
# Aggregating ball-by-ball data to one row per batsman per match
# Computing only match-level statistics without using future information
batting_match = df.groupby(
    ["matchid", "date", "inning", "batsman", "batting_team", "bowling_team"],
    as_index=False
).agg(
    runs_scored=("batsman_runs", "sum"),
    balls_faced=("iswide", lambda x: (x == 0).sum()),
    fours=("batsman_runs", lambda x: (x == 4).sum()),
    sixes=("batsman_runs", lambda x: (x == 6).sum()),
    dismissed=("is_wicket", "max")
)

batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16439 entries, 0 to 16438
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   matchid       16439 non-null  int64         
 1   date          16439 non-null  datetime64[ns]
 2   inning        16439 non-null  int64         
 3   batsman       16439 non-null  object        
 4   batting_team  16439 non-null  object        
 5   bowling_team  16439 non-null  object        
 6   runs_scored   16439 non-null  int64         
 7   balls_faced   16439 non-null  int64         
 8   fours         16439 non-null  int64         
 9   sixes         16439 non-null  int64         
 10  dismissed     16439 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(3)
memory usage: 1.4+ MB


,matchid,date,inning,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed
0,335982,2008-04-18,1,bb mccullum,kolkata knight riders,royal challengers bangalore,145,69,10,11,0
1,335982,2008-04-18,1,dj hussey,kolkata knight riders,royal challengers bangalore,12,12,1,0,1
2,335982,2008-04-18,1,mohammad hafeez,kolkata knight riders,royal challengers bangalore,5,3,1,0,0
3,335982,2008-04-18,1,rt ponting,kolkata knight riders,royal challengers bangalore,20,20,1,1,1
4,335982,2008-04-18,1,sc ganguly,kolkata knight riders,royal challengers bangalore,10,12,2,0,1


In [28]:
# Sorting matches for each batsman to enable cumulative and rolling features
batting_match = batting_match.sort_values(
    by=["batsman", "date", "matchid"]
).reset_index(drop=True)

In [29]:
# Calculating strike rate to measure scoring speed in the match
batting_match["strike_rate"] = np.where(
    batting_match["balls_faced"] > 0,
    batting_match["runs_scored"] / batting_match["balls_faced"] * 100,
    0
)

In [30]:
# Computing number of matches played before the current match
batting_match["career_matches"] = (
    batting_match.groupby("batsman").cumcount()
)

# Computing total career runs scored before the current match
batting_match["career_runs"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

# Computing total career balls faced before the current match
batting_match["career_balls"] = (
    batting_match.groupby("batsman")["balls_faced"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

# Computing career average runs per match
batting_match["career_avg_runs"] = np.where(
    batting_match["career_matches"] > 0,
    batting_match["career_runs"] / batting_match["career_matches"],
    0
)

# Computing career strike rate using past career data
batting_match["career_strike_rate"] = np.where(
    batting_match["career_balls"] > 0,
    batting_match["career_runs"] / batting_match["career_balls"] * 100,
    0
)
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16439 entries, 0 to 16438
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   matchid             16439 non-null  int64         
 1   date                16439 non-null  datetime64[ns]
 2   inning              16439 non-null  int64         
 3   batsman             16439 non-null  object        
 4   batting_team        16439 non-null  object        
 5   bowling_team        16439 non-null  object        
 6   runs_scored         16439 non-null  int64         
 7   balls_faced         16439 non-null  int64         
 8   fours               16439 non-null  int64         
 9   sixes               16439 non-null  int64         
 10  dismissed           16439 non-null  int64         
 11  strike_rate         16439 non-null  float64       
 12  career_matches      16439 non-null  int64         
 13  career_runs         16439 non-null  float64   

,matchid,date,inning,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed,strike_rate,career_matches,career_runs,career_balls,career_avg_runs,career_strike_rate
0,548346,2012-04-29,1,a ashish reddy,sunrisers hyderabad,mumbai indians,10,10,0,1,1,100.000000,0,0.0,0.0,0.00,0.0
1,548352,2012-05-04,2,a ashish reddy,sunrisers hyderabad,chennai super kings,3,3,0,0,1,100.000000,1,10.0,10.0,10.00,100.0
2,548359,2012-05-08,2,a ashish reddy,sunrisers hyderabad,punjab kings,8,8,1,0,1,100.000000,2,13.0,13.0,6.50,100.0
3,548373,2012-05-18,2,a ashish reddy,sunrisers hyderabad,rajasthan royals,10,4,2,0,0,250.000000,3,21.0,21.0,7.00,100.0
4,548376,2012-05-20,1,a ashish reddy,sunrisers hyderabad,royal challengers bangalore,4,3,0,0,0,133.333333,4,31.0,25.0,7.75,124.0


In [31]:
# Computing recent batting form using last 5 matches
batting_match["form_runs_last_5"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .shift(1)
    .rolling(window=5, min_periods=1)
    .mean()
)

# Computing recent batting form using last 10 matches
batting_match["form_runs_last_10"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .shift(1)
    .rolling(window=10, min_periods=1)
    .mean()
)

# Computing recent strike rate form using last 5 matches
batting_match["form_sr_last_5"] = (
    batting_match.groupby("batsman")["strike_rate"]
    .shift(1)
    .rolling(window=5, min_periods=1)
    .mean()
)

In [32]:
# Computing number of past matches played against the current bowling team
batting_match["matches_vs_opponent"] = (
    batting_match.groupby(["batsman", "bowling_team"]).cumcount()
)

# Computing total runs scored against the opponent before the current match
batting_match["runs_vs_opponent"] = (
    batting_match.groupby(["batsman", "bowling_team"])["runs_scored"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

# Computing average runs against the opponent using only past encounters
batting_match["avg_runs_vs_opponent"] = np.where(
    batting_match["matches_vs_opponent"] > 0,
    batting_match["runs_vs_opponent"] / batting_match["matches_vs_opponent"],
    0
)
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16439 entries, 0 to 16438
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   matchid               16439 non-null  int64         
 1   date                  16439 non-null  datetime64[ns]
 2   inning                16439 non-null  int64         
 3   batsman               16439 non-null  object        
 4   batting_team          16439 non-null  object        
 5   bowling_team          16439 non-null  object        
 6   runs_scored           16439 non-null  int64         
 7   balls_faced           16439 non-null  int64         
 8   fours                 16439 non-null  int64         
 9   sixes                 16439 non-null  int64         
 10  dismissed             16439 non-null  int64         
 11  strike_rate           16439 non-null  float64       
 12  career_matches        16439 non-null  int64         
 13  career_runs     

,matchid,date,inning,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,...,career_runs,career_balls,career_avg_runs,career_strike_rate,form_runs_last_5,form_runs_last_10,form_sr_last_5,matches_vs_opponent,runs_vs_opponent,avg_runs_vs_opponent
0,548346,2012-04-29,1,a ashish reddy,sunrisers hyderabad,mumbai indians,10,10,0,1,...,0.0,0.0,0.00,0.0,NaN,NaN,NaN,0,0.0,0.0
1,548352,2012-05-04,2,a ashish reddy,sunrisers hyderabad,chennai super kings,3,3,0,0,...,10.0,10.0,10.00,100.0,10.00,10.00,100.0,0,10.0,0.0
2,548359,2012-05-08,2,a ashish reddy,sunrisers hyderabad,punjab kings,8,8,1,0,...,13.0,13.0,6.50,100.0,6.50,6.50,100.0,0,3.0,0.0
3,548373,2012-05-18,2,a ashish reddy,sunrisers hyderabad,rajasthan royals,10,4,2,0,...,21.0,21.0,7.00,100.0,7.00,7.00,100.0,0,8.0,0.0
4,548376,2012-05-20,1,a ashish reddy,sunrisers hyderabad,royal challengers bangalore,4,3,0,0,...,31.0,25.0,7.75,124.0,7.75,7.75,137.5,0,10.0,0.0


In [33]:
# Creating an indicator for first innings batting
batting_match["is_first_innings"] = (
    batting_match["inning"] == 1
).astype(int)

# Creating a rookie indicator for players with very limited experience
batting_match["is_rookie"] = (
    batting_match["career_matches"] < 5
).astype(int)

In [34]:
# Defining runs scored in the match as the prediction target
batting_match["target_runs"] = batting_match["runs_scored"]

In [35]:
# Removing columns that directly leak target or intermediate calculations
features_df = batting_match.drop(
    columns=[
        "runs_scored",
        "career_runs",
        "career_balls",
        "runs_vs_opponent"
    ]
)

# Filling remaining missing values to ensure model compatibility
features_df = features_df.fillna(0)

features_df.info()
features_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16439 entries, 0 to 16438
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   matchid               16439 non-null  int64         
 1   date                  16439 non-null  datetime64[ns]
 2   inning                16439 non-null  int64         
 3   batsman               16439 non-null  object        
 4   batting_team          16439 non-null  object        
 5   bowling_team          16439 non-null  object        
 6   balls_faced           16439 non-null  int64         
 7   fours                 16439 non-null  int64         
 8   sixes                 16439 non-null  int64         
 9   dismissed             16439 non-null  int64         
 10  strike_rate           16439 non-null  float64       
 11  career_matches        16439 non-null  int64         
 12  career_avg_runs       16439 non-null  float64       
 13  career_strike_ra

,matchid,date,inning,batsman,batting_team,bowling_team,balls_faced,fours,sixes,dismissed,...,career_avg_runs,career_strike_rate,form_runs_last_5,form_runs_last_10,form_sr_last_5,matches_vs_opponent,avg_runs_vs_opponent,is_first_innings,is_rookie,target_runs
0,548346,2012-04-29,1,a ashish reddy,sunrisers hyderabad,mumbai indians,10,0,1,1,...,0.00,0.0,0.00,0.00,0.0,0,0.0,1,1,10
1,548352,2012-05-04,2,a ashish reddy,sunrisers hyderabad,chennai super kings,3,0,0,1,...,10.00,100.0,10.00,10.00,100.0,0,0.0,0,1,3
2,548359,2012-05-08,2,a ashish reddy,sunrisers hyderabad,punjab kings,8,1,0,1,...,6.50,100.0,6.50,6.50,100.0,0,0.0,0,1,8
3,548373,2012-05-18,2,a ashish reddy,sunrisers hyderabad,rajasthan royals,4,2,0,0,...,7.00,100.0,7.00,7.00,100.0,0,0.0,0,1,10
4,548376,2012-05-20,1,a ashish reddy,sunrisers hyderabad,royal challengers bangalore,3,0,0,0,...,7.75,124.0,7.75,7.75,137.5,0,0.0,1,1,4


In [36]:
# Saving the final batsman feature dataset for model training
features_df.to_csv("dataset_batting_features.csv", index=False)

print("Section 5.2 feature engineering completed successfully.")

Section 5.2 feature engineering completed successfully.
